In [3]:
!pip install dagshub mlflow lightgbm -q

In [4]:
import os
import gc
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import dagshub
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

dagshub.init(
    repo_owner="sophyrise",
    repo_name='ieee-cis-fraud-detection',
    mlflow=True
)

mlflow.set_experiment("LightGBM")
print("✅ MLflow tracking URI:", mlflow.get_tracking_uri())
print("✅ LightGBM version:", lgb.__version__)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=3d3f2fc9-48a7-4be1-8a2a-cb0332c0b248&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=bce4de89a42ba090d49a2fa374f69f22cf960acfe8a38c2fc8bc99602d24467f




Output()

Accessing as sophyrise

Initialized MLflow to track repo "sophyrise/ieee-cis-fraud-detection"

Repository sophyrise/ieee-cis-fraud-detection initialized!

✅ MLflow tracking URI: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow
✅ LightGBM version: 4.6.0


# Cleaning

In [5]:
with mlflow.start_run(run_name="LightGBM_Cleaning"):
    DATA_DIR = "/kaggle/input/competitions/ieee-fraud-detection"
    TXN_MISSING_THRESHOLD   = 0.80
    ID_MISSING_THRESHOLD    = 0.95
    NEAR_CONSTANT_THRESHOLD = 0.99

    train_txn = pd.read_csv(f"{DATA_DIR}/train_transaction.csv")
    train_id  = pd.read_csv(f"{DATA_DIR}/train_identity.csv")
    test_txn  = pd.read_csv(f"{DATA_DIR}/test_transaction.csv")
    test_id   = pd.read_csv(f"{DATA_DIR}/test_identity.csv")

    test_id.columns  = [c.replace('-', '_') for c in test_id.columns]
    train_id.columns = [c.replace('-', '_') for c in train_id.columns]

    train = train_txn.merge(train_id, on="TransactionID", how="left")
    test  = test_txn.merge(test_id,  on="TransactionID", how="left")

    y_train = train["isFraud"].copy()
    X_train = train.drop(columns=["isFraud", "TransactionID"])
    X_test  = test.drop(columns=["TransactionID"])

    del train, test, train_txn, train_id, test_txn, test_id
    gc.collect()

    id_like_cols  = [c for c in X_train.columns if c.startswith("id_") or c in ["DeviceType", "DeviceInfo"]]
    txn_like_cols = [c for c in X_train.columns if c not in id_like_cols]

    missing_ratio = X_train.isnull().mean()
    drop_txn      = [c for c in txn_like_cols if missing_ratio[c] > TXN_MISSING_THRESHOLD]
    drop_id       = [c for c in id_like_cols  if missing_ratio[c] > ID_MISSING_THRESHOLD]
    drop_missing  = drop_txn + drop_id

    X_train = X_train.drop(columns=drop_missing)
    X_test  = X_test.drop(columns=[c for c in drop_missing if c in X_test.columns])

    near_constant_cols = [
        c for c in X_train.columns
        if X_train[c].value_counts(dropna=False, normalize=True).iloc[0] > NEAR_CONSTANT_THRESHOLD
    ]
    X_train = X_train.drop(columns=near_constant_cols)
    X_test  = X_test.drop(columns=[c for c in near_constant_cols if c in X_test.columns])

    for col in X_train.columns:
        if col not in X_test.columns:
            X_test[col] = np.nan
    X_test = X_test[X_train.columns]

    mlflow.log_param("txn_missing_threshold",   TXN_MISSING_THRESHOLD)
    mlflow.log_param("id_missing_threshold",    ID_MISSING_THRESHOLD)
    mlflow.log_param("near_constant_threshold", NEAR_CONSTANT_THRESHOLD)
    mlflow.log_metric("train_rows",             int(X_train.shape[0]))
    mlflow.log_metric("test_rows",              int(X_test.shape[0]))
    mlflow.log_metric("features_after_cleaning",int(X_train.shape[1]))
    mlflow.log_metric("dropped_high_missing",   len(drop_missing))
    mlflow.log_metric("dropped_near_constant",  len(near_constant_cols))
    mlflow.log_metric("fraud_rate",             float(y_train.mean()))

    print(f"X_train: {X_train.shape}")
    print(f"X_test:  {X_test.shape}")
    print(f"Fraud rate: {y_train.mean():.4f}")

    X_train_clean = X_train
    X_test_clean  = X_test
    y_train_clean = y_train

X_train: (590540, 353)
X_test:  (506691, 353)
Fraud rate: 0.0350
🏃 View run LightGBM_Cleaning at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/7/runs/e9fc70c814474fc8b1d2007576d16d4f
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/7


# Feature Engineering

In [6]:
with mlflow.start_run(run_name="LightGBM_FeatureEngineering"):
    X_train = X_train_clean.copy()
    X_test  = X_test_clean.copy()
    y_train = y_train_clean.copy()

    # log transform — TransactionAmt is heavily right-skewed
    X_train["TransactionAmt_log"] = np.log1p(X_train["TransactionAmt"].clip(lower=0))
    X_test["TransactionAmt_log"]  = np.log1p(X_test["TransactionAmt"].clip(lower=0))

    # cyclic hour encoding — fraud has strong time-of-day pattern
    X_train["hour_sin"] = np.sin(2 * np.pi * ((X_train["TransactionDT"] // 3600) % 24) / 24)
    X_train["hour_cos"] = np.cos(2 * np.pi * ((X_train["TransactionDT"] // 3600) % 24) / 24)
    X_test["hour_sin"]  = np.sin(2 * np.pi * ((X_test["TransactionDT"]  // 3600) % 24) / 24)
    X_test["hour_cos"]  = np.cos(2 * np.pi * ((X_test["TransactionDT"]  // 3600) % 24) / 24)

    X_train = X_train.drop(columns=["TransactionDT"], errors="ignore")
    X_test  = X_test.drop(columns=["TransactionDT"],  errors="ignore")

    # LightGBM handles NaN natively — no imputation needed
    # ordinal encode categoricals
    cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()

    for c in cat_cols:
        mapping = {v: i for i, v in enumerate(X_train[c].astype(str).unique())}
        X_train[c] = X_train[c].astype(str).map(mapping).fillna(-1).astype(np.int32)
        X_test[c]  = X_test[c].astype(str).map(mapping).fillna(-1).astype(np.int32)

    X_test = X_test.reindex(columns=X_train.columns, fill_value=-1)

    mlflow.log_param("cat_encoding",        "ordinal_train_mapping")
    mlflow.log_param("amt_log_transform",   True)
    mlflow.log_param("cyclic_time_encoding",True)
    mlflow.log_param("nan_handling",        "lgbm_native")
    mlflow.log_metric("features_after_fe",  int(X_train.shape[1]))
    mlflow.log_metric("cat_cols_encoded",   len(cat_cols))

    print(f"X_train_fe: {X_train.shape}")
    print(f"X_test_fe:  {X_test.shape}")

    X_train_fe = X_train
    X_test_fe  = X_test
    y_train_fe = y_train

X_train_fe: (590540, 355)
X_test_fe:  (506691, 355)
🏃 View run LightGBM_FeatureEngineering at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/7/runs/2c0cf7ca24af48e6bd657a8d25e296ef
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/7


# Feature Selection

In [7]:
with mlflow.start_run(run_name="LightGBM_FeatureSelection"):
    X_train = X_train_fe.copy()
    X_test  = X_test_fe.copy()

    X_train = X_train.replace([np.inf, -np.inf], np.nan)
    X_test  = X_test.replace([np.inf, -np.inf],  np.nan)

    # drop constant columns
    nu = X_train.nunique(dropna=False)
    const_cols = nu[nu <= 1].index.tolist()
    X_train = X_train.drop(columns=const_cols, errors="ignore")
    X_test  = X_test.drop(columns=const_cols,  errors="ignore")

    # drop highly correlated — sampled for speed
    CORR_THRESHOLD = 0.98
    num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
    idx  = np.random.RandomState(42).choice(len(X_train), size=min(120_000, len(X_train)), replace=False)
    corr = X_train.iloc[idx][num_cols].fillna(0).corr().abs()
    upper     = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
    drop_corr = [c for c in upper.columns if (upper[c] > CORR_THRESHOLD).any()]

    X_train = X_train.drop(columns=drop_corr, errors="ignore")
    X_test  = X_test.drop(columns=drop_corr,  errors="ignore")
    X_test  = X_test.reindex(columns=X_train.columns, fill_value=np.nan)

    mlflow.log_param("corr_threshold",     CORR_THRESHOLD)
    mlflow.log_metric("dropped_const",     len(const_cols))
    mlflow.log_metric("dropped_high_corr", len(drop_corr))
    mlflow.log_metric("features_after_fs", int(X_train.shape[1]))

    print(f"X_train_fs: {X_train.shape}")
    print(f"X_test_fs:  {X_test.shape}")

    X_train_final = X_train
    X_test_final  = X_test

X_train_fs: (590540, 301)
X_test_fs:  (506691, 301)
🏃 View run LightGBM_FeatureSelection at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/7/runs/c60bd10962504950b177a7b751c5230b
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/7


# Training

In [8]:
import pickle, os

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_final, y_train_fe,
    test_size=0.2, random_state=42, stratify=y_train_fe
)

neg = int((y_tr == 0).sum())
pos = int((y_tr == 1).sum())
scale_pos_weight = round(neg / pos, 2)
print(f"Train: {X_tr.shape} | Val: {X_val.shape}")
print(f"scale_pos_weight = {scale_pos_weight}  (neg/pos = {neg}/{pos})")

CHECKPOINT_DIR = "/kaggle/working/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

with open(f"{CHECKPOINT_DIR}/X_train_final_lgbm.pkl", "wb") as f:
    pickle.dump(X_train_final, f)
with open(f"{CHECKPOINT_DIR}/X_test_final_lgbm.pkl", "wb") as f:
    pickle.dump(X_test_final, f)
with open(f"{CHECKPOINT_DIR}/y_train_fe_lgbm.pkl", "wb") as f:
    pickle.dump(y_train_fe, f)

print("✅ Checkpoint saved")

Train: (472432, 301) | Val: (118108, 301)
scale_pos_weight = 27.58  (neg/pos = 455902/16530)
✅ Checkpoint saved


In [9]:
import pickle, os

CHECKPOINT_DIR = "/kaggle/working/checkpoints"

with open(f"{CHECKPOINT_DIR}/X_train_final_lgbm.pkl", "rb") as f:
    X_train_final = pickle.load(f)
with open(f"{CHECKPOINT_DIR}/X_test_final_lgbm.pkl", "rb") as f:
    X_test_final = pickle.load(f)
with open(f"{CHECKPOINT_DIR}/y_train_fe_lgbm.pkl", "rb") as f:
    y_train_fe = pickle.load(f)

print("✅ Loaded from checkpoint")
print(f"X_train: {X_train_final.shape} | X_test: {X_test_final.shape}")

✅ Loaded from checkpoint
X_train: (590540, 301) | X_test: (506691, 301)


In [10]:
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_final, y_train_fe,
    test_size=0.2, random_state=42, stratify=y_train_fe
)

neg = int((y_tr == 0).sum())
pos = int((y_tr == 1).sum())
scale_pos_weight = round(neg / pos, 2)
print(f"Train: {X_tr.shape} | Val: {X_val.shape}")
print(f"scale_pos_weight = {scale_pos_weight}")

Train: (472432, 301) | Val: (118108, 301)
scale_pos_weight = 27.58


In [11]:
with mlflow.start_run(run_name="LGB_Baseline"):
    mlflow.log_param("n_estimators",     100)
    mlflow.log_param("max_depth",        6)
    mlflow.log_param("learning_rate",    0.1)
    mlflow.log_param("scale_pos_weight", 1)
    mlflow.log_param("note",             "default params, no class balancing")

    clf = lgb.LGBMClassifier(
        n_estimators=100, max_depth=6, learning_rate=0.1,
        scale_pos_weight=1, random_state=42, n_jobs=-1, verbose=-1
    )
    clf.fit(X_tr, y_tr)

    train_auc = roc_auc_score(y_tr,  clf.predict_proba(X_tr)[:, 1])
    val_auc   = roc_auc_score(y_val, clf.predict_proba(X_val)[:, 1])
    gap = train_auc - val_auc

    mlflow.log_metric("train_auc",   round(train_auc, 5))
    mlflow.log_metric("val_auc",     round(val_auc,   5))
    mlflow.log_metric("overfit_gap", round(gap, 5))

    print(f"[Baseline] Train: {train_auc:.4f} | Val: {val_auc:.4f} | Gap: {gap:.4f}")
    print(f"  -> {'OVERFIT' if gap > 0.02 else 'OK'}")

[Baseline] Train: 0.9288 | Val: 0.9192 | Gap: 0.0097
  -> OK
🏃 View run LGB_Baseline at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/7/runs/b97da8e6a7b7438998bc9a6d39beec14
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/7


In [12]:
for n_est in [100, 300, 500]:
    with mlflow.start_run(run_name=f"LGB_nestimators_{n_est}"):
        mlflow.log_param("n_estimators",     n_est)
        mlflow.log_param("max_depth",        6)
        mlflow.log_param("learning_rate",    0.1)
        mlflow.log_param("scale_pos_weight", scale_pos_weight)

        clf = lgb.LGBMClassifier(
            n_estimators=n_est, max_depth=6, learning_rate=0.1,
            scale_pos_weight=scale_pos_weight,
            random_state=42, n_jobs=-1, verbose=-1
        )
        clf.fit(X_tr, y_tr)

        train_auc = roc_auc_score(y_tr,  clf.predict_proba(X_tr)[:, 1])
        val_auc   = roc_auc_score(y_val, clf.predict_proba(X_val)[:, 1])
        gap = train_auc - val_auc

        mlflow.log_metric("train_auc",   round(train_auc, 5))
        mlflow.log_metric("val_auc",     round(val_auc,   5))
        mlflow.log_metric("overfit_gap", round(gap, 5))

        status = "OVERFIT" if gap > 0.025 else ("UNDERFIT" if val_auc < 0.90 else "OK")
        print(f"  n_estimators={n_est:<4} | Train: {train_auc:.4f} | Val: {val_auc:.4f} | Gap: {gap:.4f} [{status}]")

  n_estimators=100  | Train: 0.9331 | Val: 0.9203 | Gap: 0.0128 [OK]
🏃 View run LGB_nestimators_100 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/7/runs/1bda60487a3c43f2825e8f30e0dfb9ca
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/7
  n_estimators=300  | Train: 0.9642 | Val: 0.9431 | Gap: 0.0211 [OK]
🏃 View run LGB_nestimators_300 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/7/runs/81d7cfb8c9cc43ba814ef92d3453643d
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/7
  n_estimators=500  | Train: 0.9772 | Val: 0.9519 | Gap: 0.0253 [OVERFIT]
🏃 View run LGB_nestimators_500 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/7/runs/a363aca6237249aca1a2d49b640dd7f9
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/7


In [13]:
depth_results = []
for depth in [3, 5, 7, 10, -1]:
    label = str(depth)
    with mlflow.start_run(run_name=f"LGB_depth_{label}"):
        mlflow.log_param("n_estimators",     300)
        mlflow.log_param("max_depth",        label)
        mlflow.log_param("learning_rate",    0.1)
        mlflow.log_param("scale_pos_weight", scale_pos_weight)

        clf = lgb.LGBMClassifier(
            n_estimators=300, max_depth=depth, learning_rate=0.1,
            scale_pos_weight=scale_pos_weight,
            random_state=42, n_jobs=-1, verbose=-1
        )
        clf.fit(X_tr, y_tr)

        train_auc = roc_auc_score(y_tr,  clf.predict_proba(X_tr)[:, 1])
        val_auc   = roc_auc_score(y_val, clf.predict_proba(X_val)[:, 1])
        gap = train_auc - val_auc

        mlflow.log_metric("train_auc",   round(train_auc, 5))
        mlflow.log_metric("val_auc",     round(val_auc,   5))
        mlflow.log_metric("overfit_gap", round(gap, 5))

        if gap < 0.01:
            diagnosis = "UNDERFIT"
        elif gap > 0.03:
            diagnosis = "OVERFIT"
        else:
            diagnosis = "OK"

        depth_results.append({"max_depth": label, "train_auc": train_auc,
                               "val_auc": val_auc, "gap": gap})
        print(f"  depth={label:<4} | Train: {train_auc:.4f} | Val: {val_auc:.4f} | Gap: {gap:.4f} — {diagnosis}")

depth_df = pd.DataFrame(depth_results).set_index("max_depth")
best_depth_label = depth_df["val_auc"].idxmax()
best_depth_val   = int(best_depth_label)
print(f"\n-> Best depth: {best_depth_label}")
print(depth_df.to_string())

  depth=3    | Train: 0.9173 | Val: 0.9082 | Gap: 0.0091 — UNDERFIT
🏃 View run LGB_depth_3 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/7/runs/d64681e96897459e8784f7162b137ae7
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/7
  depth=5    | Train: 0.9537 | Val: 0.9351 | Gap: 0.0186 — OK
🏃 View run LGB_depth_5 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/7/runs/70a5a9ed011c4513af7138a5a64ad1b7
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/7
  depth=7    | Train: 0.9677 | Val: 0.9458 | Gap: 0.0219 — OK
🏃 View run LGB_depth_7 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/7/runs/c4f2fea211d84e2e8733c1f3a3a8fab9
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/7
  depth=10   | Train: 0.9713 | Val: 0.9479 | Gap: 0.0235 — OK
🏃 View run LGB_de

In [15]:
leaves_results = []
for num_leaves in [31, 63, 127, 255]:
    with mlflow.start_run(run_name=f"LGB_numleaves_{num_leaves}"):
        mlflow.log_param("n_estimators",     300)
        mlflow.log_param("max_depth",        best_depth_label)
        mlflow.log_param("num_leaves",       num_leaves)
        mlflow.log_param("learning_rate",    0.1)
        mlflow.log_param("scale_pos_weight", scale_pos_weight)

        clf = lgb.LGBMClassifier(
            n_estimators=300, max_depth=best_depth_val,
            num_leaves=num_leaves, learning_rate=0.1,
            scale_pos_weight=scale_pos_weight,
            random_state=42, n_jobs=-1, verbose=-1
        )
        clf.fit(X_tr, y_tr)

        train_auc = roc_auc_score(y_tr,  clf.predict_proba(X_tr)[:, 1])
        val_auc   = roc_auc_score(y_val, clf.predict_proba(X_val)[:, 1])
        gap = train_auc - val_auc

        mlflow.log_metric("train_auc",   round(train_auc, 5))
        mlflow.log_metric("val_auc",     round(val_auc,   5))
        mlflow.log_metric("overfit_gap", round(gap, 5))

        leaves_results.append({"num_leaves": num_leaves, "train_auc": train_auc,
                                "val_auc": val_auc, "gap": gap})
        print(f"  num_leaves={num_leaves:<4} | Train: {train_auc:.4f} | Val: {val_auc:.4f} | Gap: {gap:.4f}")

leaves_df = pd.DataFrame(leaves_results).set_index("num_leaves")
best_num_leaves = int(leaves_df["val_auc"].idxmax())
print(f"\n-> Best num_leaves: {best_num_leaves}")
print(leaves_df.to_string())

  num_leaves=31   | Train: 0.9713 | Val: 0.9479 | Gap: 0.0235
🏃 View run LGB_numleaves_31 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/7/runs/7f70e7917b4e4ca7aca15ce62ea1f16d
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/7
  num_leaves=63   | Train: 0.9873 | Val: 0.9587 | Gap: 0.0286
🏃 View run LGB_numleaves_63 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/7/runs/65ef949ea4bf4dda936204a6de336855
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/7
  num_leaves=127  | Train: 0.9949 | Val: 0.9622 | Gap: 0.0327
🏃 View run LGB_numleaves_127 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/7/runs/5f3d95844950474484564f79301c07d6
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/7
  num_leaves=255  | Train: 0.9976 | Val: 0.9630 | Gap: 0.0346
🏃 View 

In [17]:
reg_results = []
for reg_alpha, reg_lambda in [(0, 0), (0.1, 0.1), (1, 1), (0, 5), (1, 5)]:
    label = f"a{reg_alpha}_l{reg_lambda}"
    with mlflow.start_run(run_name=f"LGB_reg_{label}"):
        mlflow.log_param("n_estimators",     300)
        mlflow.log_param("max_depth",        best_depth_label)
        mlflow.log_param("num_leaves",       best_num_leaves)
        mlflow.log_param("learning_rate",    0.1)
        mlflow.log_param("reg_alpha",        reg_alpha)
        mlflow.log_param("reg_lambda",       reg_lambda)
        mlflow.log_param("scale_pos_weight", scale_pos_weight)

        clf = lgb.LGBMClassifier(
            n_estimators=300, max_depth=best_depth_val,
            num_leaves=best_num_leaves, learning_rate=0.1,
            reg_alpha=reg_alpha, reg_lambda=reg_lambda,
            scale_pos_weight=scale_pos_weight,
            random_state=42, n_jobs=-1, verbose=-1
        )
        clf.fit(X_tr, y_tr)

        train_auc = roc_auc_score(y_tr,  clf.predict_proba(X_tr)[:, 1])
        val_auc   = roc_auc_score(y_val, clf.predict_proba(X_val)[:, 1])
        gap = train_auc - val_auc

        mlflow.log_metric("train_auc",   round(train_auc, 5))
        mlflow.log_metric("val_auc",     round(val_auc,   5))
        mlflow.log_metric("overfit_gap", round(gap, 5))

        reg_results.append({"reg_alpha": reg_alpha, "reg_lambda": reg_lambda,
                             "train_auc": train_auc, "val_auc": val_auc, "gap": gap})
        print(f"  alpha={reg_alpha} lambda={reg_lambda} | Train: {train_auc:.4f} | Val: {val_auc:.4f} | Gap: {gap:.4f}")

reg_df = pd.DataFrame(reg_results)
best_reg = reg_df.loc[reg_df["val_auc"].idxmax()]
best_alpha  = best_reg["reg_alpha"]
best_lambda = best_reg["reg_lambda"]
print(f"\n-> Best reg: alpha={best_alpha}, lambda={best_lambda}")
print(reg_df.to_string())

  alpha=0 lambda=0 | Train: 0.9976 | Val: 0.9630 | Gap: 0.0346
🏃 View run LGB_reg_a0_l0 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/7/runs/bba56f675afb46b8855ac409057d30c9
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/7
  alpha=0.1 lambda=0.1 | Train: 0.9978 | Val: 0.9636 | Gap: 0.0342
🏃 View run LGB_reg_a0.1_l0.1 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/7/runs/699e2ce45b0644b2b2f4647b05a4f779
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/7
  alpha=1 lambda=1 | Train: 0.9974 | Val: 0.9628 | Gap: 0.0347
🏃 View run LGB_reg_a1_l1 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/7/runs/09707e4f52194924870212920a489065
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/7
  alpha=0 lambda=5 | Train: 0.9967 | Val: 0.9637 | Gap: 0.0330
🏃 Vie

In [18]:
lr_results = []
for lr in [0.01, 0.05, 0.1]:
    n_est = 1000 if lr == 0.01 else (500 if lr == 0.05 else 300)
    
    with mlflow.start_run(run_name=f"LGB_lr_{lr}"):
        mlflow.log_param("learning_rate", lr)
        mlflow.log_param("n_estimators", n_est)
        mlflow.log_param("max_depth", best_depth_val)
        mlflow.log_param("num_leaves", best_num_leaves)
        
        clf = lgb.LGBMClassifier(
            n_estimators=n_est,
            learning_rate=lr,
            max_depth=best_depth_val,
            num_leaves=best_num_leaves,
            scale_pos_weight=scale_pos_weight,
            random_state=42,
            n_jobs=-1,
            verbose=-1
        )
        clf.fit(X_tr, y_tr)
        
        train_auc = roc_auc_score(y_tr, clf.predict_proba(X_tr)[:, 1])
        val_auc   = roc_auc_score(y_val, clf.predict_proba(X_val)[:, 1])
        gap = train_auc - val_auc
        
        mlflow.log_metric("train_auc", round(train_auc, 5))
        mlflow.log_metric("val_auc", round(val_auc, 5))
        mlflow.log_metric("overfit_gap", round(gap, 5))
        
        lr_results.append({"lr": lr, "n_est": n_est, "train_auc": train_auc, "val_auc": val_auc, "gap": gap})
        print(f"  LR={lr:<5} | N={n_est:<4} | Val AUC: {val_auc:.4f} | Gap: {gap:.4f}")

  LR=0.01  | N=1000 | Val AUC: 0.9528 | Gap: 0.0320
🏃 View run LGB_lr_0.01 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/7/runs/482d0c4d0a174e91b746183e0bf4ddc7
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/7
  LR=0.05  | N=500  | Val AUC: 0.9628 | Gap: 0.0333
🏃 View run LGB_lr_0.05 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/7/runs/bb093338d1b14e7497b833e6916bc207
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/7
  LR=0.1   | N=300  | Val AUC: 0.9630 | Gap: 0.0346
🏃 View run LGB_lr_0.1 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/7/runs/c7ab72bc81f64cd89e4627ad45a859b7
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/7


In [19]:
lr_df = pd.DataFrame(lr_results)
best_lr_row = lr_df.loc[lr_df['val_auc'].idxmax()]

best_lr = best_lr_row['lr']
best_n_estimators = int(best_lr_row['n_est'])

In [20]:
sample_idx = np.random.RandomState(42).choice(len(X_train_final), size=150_000, replace=False)
X_cv = X_train_final.iloc[sample_idx]
y_cv = y_train_fe.iloc[sample_idx]

with mlflow.start_run(run_name="LGB_CrossValidation_5fold"):
    mlflow.log_param("n_estimators",    best_n_estimators)
    mlflow.log_param("learning_rate",   best_lr)
    mlflow.log_param("max_depth",       best_depth_label)
    mlflow.log_param("num_leaves",      best_num_leaves)
    mlflow.log_param("reg_alpha",       best_alpha)
    mlflow.log_param("reg_lambda",      best_lambda)
    mlflow.log_param("scale_pos_weight", scale_pos_weight)

    clf_cv = lgb.LGBMClassifier(
        n_estimators=best_n_estimators, 
        learning_rate=best_lr,
        max_depth=best_depth_val,
        num_leaves=best_num_leaves, 
        reg_alpha=best_alpha, 
        reg_lambda=best_lambda,
        scale_pos_weight=scale_pos_weight,
        random_state=42, n_jobs=-1, verbose=-1
    )

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(clf_cv, X_cv, y_cv, cv=cv, scoring="roc_auc")

    for i, score in enumerate(cv_scores):
        mlflow.log_metric("fold_auc", round(score, 5), step=i)

    mlflow.log_metric("cv_mean_auc", round(cv_scores.mean(), 5))
    mlflow.log_metric("cv_std_auc",  round(cv_scores.std(),  5))

    print(f"CV folds: {[round(s, 4) for s in cv_scores]}")
    print(f"Mean: {cv_scores.mean():.4f} | Std: {cv_scores.std():.4f}")
    print(f"  -> {'STABLE' if cv_scores.std() < 0.005 else 'HIGH VARIANCE'}")

CV folds: [np.float64(0.9283), np.float64(0.9355), np.float64(0.9286), np.float64(0.9313), np.float64(0.9244)]
Mean: 0.9296 | Std: 0.0037
  -> STABLE
🏃 View run LGB_CrossValidation_5fold at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/7/runs/5f6649230e90417dadf6c1f6b1b3870d
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/7


In [21]:
X_tr_f, X_val_f, y_tr_f, y_val_f = train_test_split(
    X_train_final, y_train_fe,
    test_size=0.2, random_state=42, stratify=y_train_fe
)

neg_f = int((y_tr_f == 0).sum())
pos_f = int((y_tr_f == 1).sum())
spw_f = round(neg_f / pos_f, 2)

with mlflow.start_run(run_name="LGB_Final_Pipeline") as run:
    mlflow.log_param("n_estimators",    best_n_estimators)
    mlflow.log_param("learning_rate",   best_lr)
    mlflow.log_param("max_depth",       best_depth_label)
    mlflow.log_param("num_leaves",      best_num_leaves)
    mlflow.log_param("reg_alpha",       best_alpha)
    mlflow.log_param("reg_lambda",      best_lambda)
    mlflow.log_param("scale_pos_weight", spw_f)
    mlflow.log_param("note", f"Based on sweep results: lr={best_lr}, n_est={best_n_estimators}")

    final_clf = lgb.LGBMClassifier(
        n_estimators=best_n_estimators, 
        learning_rate=best_lr,
        max_depth=best_depth_val,
        num_leaves=best_num_leaves, 
        reg_alpha=best_alpha, 
        reg_lambda=best_lambda,
        scale_pos_weight=spw_f,
        random_state=42, n_jobs=-1, verbose=-1
    )

    final_clf.fit(X_tr_f, y_tr_f)

    train_auc = roc_auc_score(y_tr_f,  final_clf.predict_proba(X_tr_f)[:, 1])
    val_auc   = roc_auc_score(y_val_f, final_clf.predict_proba(X_val_f)[:, 1])
    gap = train_auc - val_auc

    mlflow.log_metric("train_auc",   round(train_auc, 5))
    mlflow.log_metric("val_auc",     round(val_auc,   5))
    mlflow.log_metric("overfit_gap", round(gap, 5))

    final_pipe = Pipeline([("clf", final_clf)])

    mlflow.sklearn.log_model(
        sk_model=final_pipe,
        artifact_path="lgbm_pipeline",
        registered_model_name="LightGBM_FraudDetection"
    )

    pkl_path = "/kaggle/working/lgbm_final_pipeline.pkl"
    with open(pkl_path, "wb") as f:
        pickle.dump(final_pipe, f)
    mlflow.log_artifact(pkl_path)

    print(f"[Final] Train: {train_auc:.4f} | Val: {val_auc:.4f} | Gap: {gap:.4f}")
    print(f"  -> {'OVERFIT' if gap > 0.03 else 'OK'}")
    print(f"Run ID: {run.info.run_id}")
    print("✅ Registered as LightGBM_FraudDetection")

2026/05/04 20:34:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/04 20:34:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'LightGBM_FraudDetection' already exists. Creating a new version of this model...
2026/05/04 20:35:03 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: LightGBM_FraudDetection, version 2
Created version '2' of model 'LightGBM_FraudDetection'.


[Final] Train: 0.9971 | Val: 0.9639 | Gap: 0.0332
  -> OVERFIT
Run ID: 7ff9e68a7d18428f996f1dd3ebd8f167
✅ Registered as LightGBM_FraudDetection
🏃 View run LGB_Final_Pipeline at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/7/runs/7ff9e68a7d18428f996f1dd3ebd8f167
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/7
